# Phase 1 - behavioral fidelity: greedy vs sampling (no retrain)

Fresh Colab, **A100 + High-RAM strongly recommended** (CELL 2 builds `base_resized` by loading the base in fp32 on CPU ~12GB, and CELL 3 free-runs generation over the holdout twice), then **Run All**.

Reloads the P6 two-stage adapter from HF and runs `sft.score_tokens` with `--behavioral-sampling both`, so you get, on the same unit-aware holdout:
- the teacher-forced `METRIC=` (token accuracy, unchanged locked contract), and
- **behavioral fidelity** (decode -> re-measure behavior_v2 -> direction+magnitude agreement vs the requested spec) under **greedy** vs **sampling t=0.7** + collapse-rate + decoded deltaE.

This is the real test Phase 0 pointed to: does sampling recover the *requested behavior* (not just RMS strength)? Scored with `attribute_spec_text` conditioning (via `configs/candidate_two_stage.json`) to match how P6 was trained.

The only long wait is the one-time ~9.85GB corpus stage. Set `BEHAVIORAL_LIMIT = 0` in CELL 3 for the full holdout.


In [ ]:
# CELL 1 - provision the runtime (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')


In [ ]:
# CELL 2 - build base_resized (score_tokens needs it) + download the P6 two-stage adapter from HF
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    print('building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...')
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
assert vr.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f', vr.get('tokenizer_version')
print('base_resized OK; identity:', vr['tokenizer_version'])

from huggingface_hub import snapshot_download
REPO_ID = 'ericrcwu/LUT_SLM_sft_adapters'
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id=REPO_ID, allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
ADAPTER = 'models/sft_adapters/' + ADAPTER_SUBFOLDER
am = json.load(open(pathlib.Path(ADAPTER) / 'adapter_manifest.json'))
assert am['tokenizer_version'] == vr['tokenizer_version'], 'adapter/tokenizer identity mismatch'
assert pathlib.Path(ADAPTER, 'adapter_model.safetensors').is_file(), ADAPTER
print('adapter ready:', ADAPTER, '| step:', am.get('adapter_step'), '| sha:', am.get('adapter_sha256', '')[:16])


In [ ]:
# CELL 3 - score P6: teacher-forced METRIC + behavioral fidelity under greedy vs sampling t=0.7
import os, sys, json, subprocess
ADAPTER = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'
BEHAVIORAL_LIMIT = 64   # rows in the (slow) free-running pass; 0 = full unit-aware holdout

env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.score_tokens',
       '--config', 'configs/candidate_two_stage.json',   # input_field=attribute_spec_text (matches P6)
       '--resized-model', 'models/base_resized',
       '--adapter', ADAPTER,
       '--limit', '0',                                    # full holdout for the teacher-forced METRIC
       '--behavioral-sampling', 'both',
       '--behavioral-limit', str(BEHAVIORAL_LIMIT),
       '--behavioral-temperature', '0.7', '--behavioral-top-p', '0.9']
print('running:', ' '.join(cmd), '\n')
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-9000:])
if p.returncode != 0:
    print('--- STDERR tail ---\n', p.stderr[-3000:])

summary = None
for line in p.stdout.splitlines():
    if line.strip().startswith('{') and 'score_summary' in line:
        try:
            summary = json.loads(line)['score_summary']
        except Exception:
            pass
if summary:
    print('\n================= SUMMARY =================')
    print('teacher-forced token_accuracy (METRIC):', round(summary.get('metric', 0.0), 4),
          '(one-stage baseline 0.362)')
    for key, tag in (('behavioral', 'greedy'), ('behavioral_sampled', 'sample t=0.7')):
        b = summary.get(key) or {}
        if 'behavioral_fidelity_mean' in b:
            print('  %-13s fidelity=%s  collapse_rate=%s  resid_median=%s  deltaE=%s  scored=%s refused=%s' % (
                tag, b.get('behavioral_fidelity_mean'), b.get('collapse_rate'),
                b.get('residual_norm_median'), b.get('decoded_delta_e_mean'),
                b.get('scored'), b.get('refused')))
    print('\nRead: higher behavioral fidelity under sample-t0.7 than greedy => decoding regime is the')
    print('      lever (do the training fixes). Both low => input/loss are the bottleneck.')
